# CML α-sweep animation

Track a single CML system through a smooth α sweep (ε fixed) in SPI feature space.

**Design**: fit PCA on existing anchor regimes in `data/embeddings/proof_260420/` (M=20, T=2000), then project an α-sweep trajectory into that fixed space. Each frame is a self-contained SPI snapshot; initial conditions are carried across α values within a chain to give visual continuity (quasi-static sweep).

Pilot: α ∈ [1.40, 2.00] step 0.015 (~41 frames), ε = 0.3, 3 independent chains, blended_sonnet SPI config.

In [ ]:
import os
# BLAS pinning: set in shell BEFORE the kernel starts (these are no-ops if
# BLAS is already loaded). The PYSPI_N_JOBS env drives pyspi-fork worker count.
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")
os.environ["PYSPI_N_JOBS"] = "4"

import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation as manim
from matplotlib.colors import Normalize
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

ROOT = Path("/Users/wedi0306/Code/mts-spi-study-cluster")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.generators.dynamical import generate_cml_logistic
from src.compute import run_pyspi

ANCHOR_DIR = ROOT / "data/embeddings/proof_260420"
PYSPI_CONFIG = ROOT / "configs/pyspi-v2/blended_sonnet_config.yaml"
OUT_DIR = ROOT / "data/embeddings/cml_alpha_sweep_260423"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("anchor dir exists:", ANCHOR_DIR.exists())
print("pyspi config exists:", PYSPI_CONFIG.exists())

## 1. Load anchor SPI features

In [ ]:
def _spi_feature_vector(npz_like, sym_mask):
    """Flatten a run's SPI matrices into one feature vector. Symmetric SPIs
    contribute upper-triangle (M(M-1)/2), directed contribute full off-diag.
    SPIs missing from npz_like (e.g. silently dropped by pyspi at compute time)
    are filled with NaN so the output dim stays fixed across runs."""
    M = None
    feats = []
    for name, is_sym in sym_mask.items():
        if name in npz_like:
            mat = np.asarray(npz_like[name])
            if M is None:
                M = mat.shape[0]
        else:
            if M is None:
                # infer M from any other present key
                for k in npz_like:
                    M = np.asarray(npz_like[k]).shape[0]
                    break
            mat = np.full((M, M), np.nan)
        m = mat.shape[0]
        if is_sym:
            iu = np.triu_indices(m, k=1)
            feats.append(mat[iu])
        else:
            off = ~np.eye(m, dtype=bool)
            feats.append(mat[off])
    return np.concatenate(feats)

M_FIXED = 10
T_FIXED = 500

# Derive SPI names + symmetry from one anchor (anchors share the config).
_first = next(ANCHOR_DIR.glob(f"*/M{M_FIXED}_T{T_FIXED}_I*/spi_mpis.npz"))
_z = np.load(_first)
sym_mask = {n: bool(np.allclose(_z[n], _z[n].T, atol=1e-10, equal_nan=True)) for n in _z.files}
print(f"{len(sym_mask)} SPIs | {sum(sym_mask.values())} symmetric, {sum(not v for v in sym_mask.values())} directed")

In [ ]:
records = []
for class_dir in sorted(p for p in ANCHOR_DIR.iterdir() if p.is_dir()):
    for run_dir in sorted(class_dir.glob(f"M{M_FIXED}_T{T_FIXED}_I*")):
        npz_path = run_dir / "spi_mpis.npz"
        meta_path = run_dir / "meta.json"
        if not npz_path.exists() or not meta_path.exists():
            continue
        with open(meta_path) as f:
            meta = json.load(f)
        feat = _spi_feature_vector(np.load(npz_path), sym_mask)
        records.append({
            "class": class_dir.name,
            "instance": meta.get("instance_index"),
            "feat": feat,
        })

X_anchor_raw = np.stack([r["feat"] for r in records])
y_class = np.array([r["class"] for r in records])
print(f"{len(records)} anchor runs; X_anchor_raw shape: {X_anchor_raw.shape}")
print("class counts:", dict(zip(*np.unique(y_class, return_counts=True))))

## 2. NaN masking + PCA fit

In [ ]:
# Drop columns that have any non-finite value across the anchor set.
X_anchor = np.where(np.isfinite(X_anchor_raw), X_anchor_raw, np.nan)
valid_cols = ~np.isnan(X_anchor).any(axis=0)
print(f"keeping {valid_cols.sum()}/{len(valid_cols)} columns (dropped {(~valid_cols).sum()} with NaN)")
X_anchor_c = X_anchor[:, valid_cols]

scaler = StandardScaler()
X_anchor_s = scaler.fit_transform(X_anchor_c)

pca = PCA(n_components=2, random_state=0)
Z_anchor = pca.fit_transform(X_anchor_s)

print(f"PC1: {pca.explained_variance_ratio_[0]:.2%}  |  PC2: {pca.explained_variance_ratio_[1]:.2%}  |  cum: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
CML_CLASSES = {"brownian-defect", "chaotic-traveling-wave", "defect-turbulence", "fdstc",
               "frozen-chaos", "pattern-selection", "sti-i", "sti-ii",
               "traveling-wave", "traveling-wave-kaneko67"}
cml_mask = np.isin(y_class, list(CML_CLASSES))
# Refit PCA on CML anchors only; controls get projected into that space
scaler = StandardScaler().fit(X_anchor_c[cml_mask])
pca = PCA(n_components=2, random_state=0).fit(scaler.transform(X_anchor_c[cml_mask]))
Z_anchor = pca.transform(scaler.transform(X_anchor_c))


In [ ]:
# Sanity-check anchor scatter.
classes = np.unique(y_class)
cmap_disc = plt.get_cmap("tab20", len(classes))

fig, ax = plt.subplots(figsize=(10, 7))
for i, cls in enumerate(classes):
    m = y_class == cls
    ax.scatter(Z_anchor[m, 0], Z_anchor[m, 1], label=cls, s=40, color=cmap_disc(i), edgecolor="none")
ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left", fontsize=8)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title(f"Anchor embedding — {len(records)} runs (M={M_FIXED}, T={T_FIXED}, {sum(valid_cols)} features)")
plt.tight_layout()
plt.show()

## 3. Aggregate α-sweep results from cluster

Compute runs off-laptop via PBS (too expensive locally at blended_sonnet × 123 runs):

- config: `configs/generate/embeddings/cml_alpha_sweep.yaml` (41 α × 3 instances = 123 datasets, M=10, T=500, fresh ICs per job)
- job: `jobs/physics/run_cml_alpha_sweep.pbs` → submit with `qsub`
- rsync results back under `data/embeddings/cml_alpha_sweep_260423/cml-alpha-sweep/`

The aggregation cell walks dataset dirs and builds the same `sweep_feats / sweep_chains / sweep_alpha_idx` arrays the original inline sweep produced, so cells 4–6 below remain unchanged. Note: cluster path uses **fresh ICs** per (α, instance), not chained — the harness doesn't support chaining. Visual continuity is still there (CML attractors vary smoothly with α), just with more seed-jitter than chained ICs would give.

In [ ]:
ALPHA_VALUES = np.round(np.arange(1.40, 2.001, 0.015), 4)
EPS_FIXED = 0.3
SWEEP_DIR = OUT_DIR / "cml-alpha-sweep"

print(f"expected {len(ALPHA_VALUES)} α values × 3 instances = {len(ALPHA_VALUES)*3} SPI runs")
print(f"sweep dir: {SWEEP_DIR}  exists={SWEEP_DIR.exists()}")
if SWEEP_DIR.exists():
    completed = sum(1 for p in SWEEP_DIR.glob("M*_T*_I*_a*") if (p / "spi_mpis.npz").exists())
    print(f"completed datasets: {completed}")

In [ ]:
# Walk completed sweep datasets and build feature matrix in the shape
# expected downstream. `instance_index` from meta.json becomes `chain`;
# α comes from generator.params.alpha in meta.json.
ALPHA_TO_IDX = {float(a): i for i, a in enumerate(ALPHA_VALUES)}

sweep_records = []
for run_dir in sorted(SWEEP_DIR.glob("M*_T*_I*_a*")):
    npz_path = run_dir / "spi_mpis.npz"
    meta_path = run_dir / "meta.json"
    if not (npz_path.exists() and meta_path.exists()):
        continue
    with open(meta_path) as f:
        meta = json.load(f)
    alpha = float(meta["generator"]["params"]["alpha"])
    instance = int(meta["instance_index"])
    # match alpha to nearest ALPHA_VALUES entry (avoids float identity issues)
    a_idx = int(np.argmin(np.abs(ALPHA_VALUES - alpha)))
    if abs(ALPHA_VALUES[a_idx] - alpha) > 1e-6:
        print(f"  [warn] {run_dir.name}: α={alpha} not in ALPHA_VALUES grid, using closest {ALPHA_VALUES[a_idx]}")
    feat = _spi_feature_vector(np.load(npz_path), sym_mask)
    sweep_records.append({"chain": instance, "alpha_idx": a_idx, "feat": feat})

if not sweep_records:
    raise RuntimeError(
        f"No completed sweep datasets found under {SWEEP_DIR}. "
        "Submit jobs/physics/run_cml_alpha_sweep.pbs on the cluster and rsync results back first."
    )

sweep_feats = np.stack([r["feat"] for r in sweep_records])
sweep_chains = np.array([r["chain"] for r in sweep_records])
sweep_alpha_idx = np.array([r["alpha_idx"] for r in sweep_records])
print(f"aggregated {len(sweep_records)} frames | feat dim {sweep_feats.shape[1]} | "
      f"chains {sorted(set(sweep_chains.tolist()))} | α-coverage {len(set(sweep_alpha_idx.tolist()))}/{len(ALPHA_VALUES)}")

## 4. Project sweep into anchor PCA space

In [ ]:
Xs = np.where(np.isfinite(sweep_feats), sweep_feats, np.nan)
Xs = Xs[:, valid_cols]

# Impute any residual NaN in sweep columns with anchor column mean,
# so a single bad SPI run doesn't kill a whole frame.
col_mean = np.nanmean(X_anchor_c, axis=0)
nan_mask = np.isnan(Xs)
if nan_mask.any():
    frames_with_nan = np.where(nan_mask.any(axis=1))[0]
    print(f"imputing {nan_mask.sum()} NaN cells across {len(frames_with_nan)} frames")
    Xs = np.where(nan_mask, col_mean[None, :], Xs)

Xs_s = scaler.transform(Xs)
Z_sweep = pca.transform(Xs_s)
print("Z_sweep shape:", Z_sweep.shape)

## 5. Static trajectory plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
# Anchors (muted)
for i, cls in enumerate(classes):
    m = y_class == cls
    ax.scatter(Z_anchor[m, 0], Z_anchor[m, 1], label=cls, s=30, color=cmap_disc(i), alpha=0.55, edgecolor="none")

# Sweep: one polyline per chain (ordered by alpha_idx), points colored by α
norm = Normalize(vmin=float(ALPHA_VALUES.min()), vmax=float(ALPHA_VALUES.max()))
alpha_per_frame = ALPHA_VALUES[sweep_alpha_idx]
n_chains_actual = int(sweep_chains.max()) + 1
for c in range(n_chains_actual):
    m = sweep_chains == c
    # sort by alpha_idx for clean trajectory
    order = np.argsort(sweep_alpha_idx[m])
    ax.plot(Z_sweep[m, 0][order], Z_sweep[m, 1][order], color="black", lw=0.5, alpha=0.4)
sc = ax.scatter(Z_sweep[:, 0], Z_sweep[:, 1], c=alpha_per_frame, cmap="viridis", norm=norm,
                s=35, edgecolor="black", lw=0.3, zorder=5)
fig.colorbar(sc, ax=ax, label="α")
ax.legend(bbox_to_anchor=(1.28, 1.0), loc="upper left", fontsize=7)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title(f"α-sweep trajectory (ε={EPS_FIXED}, {n_chains_actual} chains × {len(ALPHA_VALUES)} α values)")
plt.tight_layout()
fig.savefig(OUT_DIR / "static_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Animation

In [ ]:
N_ALPHA = len(ALPHA_VALUES)
# (n_alpha, n_chains, 2): trajectory matrix indexed by alpha_idx and chain
Z_by_alpha = np.full((N_ALPHA, n_chains_actual, 2), np.nan)
for i in range(len(Z_sweep)):
    Z_by_alpha[sweep_alpha_idx[i], sweep_chains[i]] = Z_sweep[i]

fig, ax = plt.subplots(figsize=(10, 7))
for i, cls in enumerate(classes):
    m = y_class == cls
    ax.scatter(Z_anchor[m, 0], Z_anchor[m, 1], label=cls, s=25, color=cmap_disc(i), alpha=0.35, edgecolor="none")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left", fontsize=7)

# Fix axes to the full sweep extent so the camera doesn't jump around.
xlim = (min(Z_anchor[:, 0].min(), Z_sweep[:, 0].min()) - 0.5,
        max(Z_anchor[:, 0].max(), Z_sweep[:, 0].max()) + 0.5)
ylim = (min(Z_anchor[:, 1].min(), Z_sweep[:, 1].min()) - 0.5,
        max(Z_anchor[:, 1].max(), Z_sweep[:, 1].max()) + 0.5)
ax.set_xlim(xlim)
ax.set_ylim(ylim)

trail_lines = [ax.plot([], [], color="black", lw=1, alpha=0.55)[0] for _ in range(n_chains_actual)]
heads = ax.scatter([], [], s=120, c=[], cmap="viridis", norm=norm, edgecolor="black", lw=0.9, zorder=6)
title = ax.set_title("")

def _init():
    for ln in trail_lines:
        ln.set_data([], [])
    heads.set_offsets(np.empty((0, 2)))
    heads.set_array(np.empty((0,)))
    return [*trail_lines, heads, title]

def _update(frame):
    for c in range(n_chains_actual):
        xs = Z_by_alpha[: frame + 1, c, 0]
        ys = Z_by_alpha[: frame + 1, c, 1]
        trail_lines[c].set_data(xs, ys)
    pts = Z_by_alpha[frame]
    mask = ~np.isnan(pts[:, 0])
    heads.set_offsets(pts[mask])
    heads.set_array(np.full(mask.sum(), float(ALPHA_VALUES[frame])))
    title.set_text(f"α = {ALPHA_VALUES[frame]:.3f}  (ε = {EPS_FIXED}, frame {frame+1}/{N_ALPHA})")
    return [*trail_lines, heads, title]

anim = manim.FuncAnimation(fig, _update, frames=N_ALPHA, init_func=_init, interval=200, blit=False)
out_mp4 = OUT_DIR / "alpha_sweep.mp4"
try:
    anim.save(out_mp4, fps=5, dpi=150)
    print(f"saved {out_mp4}")
except Exception as exc:
    print(f"mp4 save failed ({exc}); falling back to gif")
    out_gif = OUT_DIR / "alpha_sweep.gif"
    anim.save(out_gif, writer="pillow", fps=5, dpi=100)
    print(f"saved {out_gif}")
plt.close(fig)